In [13]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set seaborn style for beautiful visualizations
sns.set_theme(style="whitegrid")

# VISUALIZE THE DATA

def load_data():
    file_path="../data/processed/bitcoin_processed.csv"
    if os.path.exists(file_path):
        return pd.read_csv(file_path)
    else:
        raise FileNotFoundError(f"Error: {file_path}")
    
def save_plot(filename):
    output_dir = "../outputs/charts"
    os.makedirs(output_dir, exist_ok=True)
    full_path = os.path.join(output_dir, filename)
    plt.savefig(full_path, bbox_inches='tight', dpi=300)
    plt.close()

In [14]:
# Does fear increase as prices fall?
def plot_price_vs_fear(df):
    fig, ax1 = plt.subplots(figsize=(14, 6))
    ax2 = ax1.twinx()

    sns.lineplot(data=df, x="date", y="price", ax=ax1, color="blue", label="Price")
    sns.lineplot(data=df, x="date", y="fear_greed_value", ax=ax2, color="red", label="Fear & Greed")

    ax1.set_ylabel("Price ", color="blue")
    ax2.set_ylabel("Fear & Greed Value", color="red")
    ax1.set_xlabel("Date")
    plt.title("Bitcoin Price vs Fear & Greed Index")

    # Align legends together
    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")
    if ax2.get_legend():
        ax2.get_legend().remove()

    plt.tight_layout()
    save_plot("1_price_vs_fear_greed.png") 
    plt.show()

In [15]:
# correlation heatmap
def plot_market_stress_heatmap(df):
    plt.figure(figsize=(12,6))

    cols=['price','daily_return','volatility','volume_spike','fear_greed_value','sentiment_negativity','market_stress_index','panic_score']
    corr_matrix=df[cols].corr()

    sns.heatmap(corr_matrix,annot=True , cmap='coolwarm',fmt=".2f",linewidths=0.5)
    plt.title("Correlation Matrix of Market Stress" , fontweight="bold")
    save_plot("2_heatmap.png")
    plt.show()

In [16]:
def plot_panic_vs_daily_return(df):
    plt.figure(figsize=(12,6))
    sns.scatterplot(
        data=df,
        x="date",
        y="daily_return",
        hue="is_panic_day",
        palette={0: '#3498db', 1: '#e74c3c'}
    )
    plt.title("Panic Days vs Daily Return", fontweight='bold')
    plt.xlabel("Date")
    plt.ylabel("Daily Return")
    plt.legend(title="Panic Day")
    save_plot('3_panic_vs_daily_return.png')
    plt.show()

In [17]:
def plot_sentiment_vs_return(df):
    # Filter to days that actually have news to prevent a vertical pileup at zero
    df_news = df[df["headline_count"].notna() & (df["headline_count"] > 0)]
    
    plt.figure(figsize=(12,6))
    if not df_news.empty:
        sns.scatterplot(
            data=df_news,
            x="sentiment_negativity", 
            y="next_day_return",
            color="#34495e"
        )
    else:
        plt.text(0.5, 0.5, "No News Headlines for Plotting", ha="center", va="center")
        
    plt.title("Sentiment Negativity vs Next Day Return", fontweight='bold')
    plt.xlabel("Sentiment Negativity")
    plt.ylabel('Next Day Return')
    save_plot("4_sentiment_vs_return.png")
    plt.show()

In [18]:
try:
    df=load_data()
    plot_price_vs_fear(df)
    plot_market_stress_heatmap(df)
    plot_panic_vs_daily_return(df)
    plot_sentiment_vs_return(df)

    print("=== ALL PLOTS WERE SAVED ===")

except Exception as e:
    print(f"An error: {e}")

=== ALL PLOTS WERE SAVED ===
